### DP-SGD Implementation

We train a class-conditional DDPM on MNIST under DP-SGD and evaluate the privacy-utility tradeoff across different epsilon budgets.

The model is a UNet that predicts the v-target ($v = \sqrt{\bar\alpha}\,\epsilon - \sqrt{1-\bar\alpha}\,x_0$) instead of raw noise. Class conditioning is injected by summing label embeddings with the timestep embedding. An 11th null class is included to support classifier-free guidance at inference.

The main technique for improving the privacy-utility tradeoff is noise multiplicity: each training step reuses the same batch K times with independently drawn noise, averaging the K losses before the backward pass to achieve the reduces variance of the gradient. This gives more gradient signal per unit of privacy budget without additional epsilon cost.

EMA keeps a running average of model weights and is used for sampling instead of the raw weights, which helps smooth out the instability introduced by gradient clipping and noise addition.

On the privacy side, Opacus clips per-sample gradients to norm C, adds Gaussian noise proportional to σ·C, and uses Poisson sampling for tighter RDP accounting. 

##### Evaluation Pipeline
After training, each model is evaluated on three axes:

| Metric | Tool | What it measures |
|---|---|---|
| **FID** (Fréchet Inception Distance) | `torch_fidelity` | Distribution-level similarity between 20 k generated and 20 k real MNIST images |
| **IS** (Inception Score) | `torch_fidelity` | Sharpness and diversity of generated images |
| **Downstream NN accuracy** | MLP + CNN trained on 60 k synthetic images, tested on real MNIST test set | Whether synthetic data is useful for training classifiers |
| **Downstream sklearn accuracy** | LogisticRegression, DecisionTree, RandomForest | Same downstream utility via classical classifiers |


In [ ]:
import torch
print(torch.cuda.device_count())      
print(torch.cuda.current_device())    
print(torch.cuda.get_device_name(0))  

1
0
NVIDIA A100-SXM4-80GB


In [ ]:
from google.colab import drive
import os
from datetime import datetime
import math
import json
import random
import numpy as np
import pandas as pd
from tqdm import tqdm

# import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as transforms
from torchvision.datasets import MNIST
from torch.utils.data import DataLoader
from torchvision.utils import save_image, make_grid

import matplotlib.pyplot as plt
from opacus import PrivacyEngine
from opacus.validators import ModuleValidator
from torch_fidelity import calculate_metrics

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import torch.nn.utils.parametrize as parametrize
from opacus.grad_sample import GradSampleModule

In [ ]:
def seed_all(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_all(42)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

device: cuda


In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE_DRIVE_DIR = "/content/drive/MyDrive/diffusion_experiments"
except Exception:
    BASE_DRIVE_DIR = "./diffusion_experiments"

DATE_STR = datetime.now().strftime("%d.%m.%Y")
RUN_DIR = os.path.join(BASE_DRIVE_DIR, DATE_STR)
os.makedirs(RUN_DIR, exist_ok=True)
print("RUN_DIR:", RUN_DIR)

Mounted at /content/drive
All outputs will be saved to: /content/drive/MyDrive/diffusion_experiments/14.01.2026


In [ ]:
def make_exp_dir(exp_name: str):
    exp_dir = os.path.join(RUN_DIR, exp_name)
    os.makedirs(exp_dir, exist_ok=True)
    for sub in ["checkpoints", "samples", "metrics", "fid_is", "synthetic_data", "diagnostics"]:
        os.makedirs(os.path.join(exp_dir, sub), exist_ok=True)
    return exp_dir

def save_json(obj, path):
    with open(path, "w") as f:
        json.dump(obj, f, indent=2)

### Data

In [ ]:
# Rescale pixel values from [0,1] → [-1,1] so the diffusion model output
# spans the same range as standard Gaussian noise used during the forward process.
train_tfm = transforms.Compose([
    transforms.ToTensor(),
    transforms.Lambda(lambda x: (x * 2) - 1),  # [-1, 1]
])
test_tfm = transforms.Compose([
    transforms.ToTensor(),
    transforms.Lambda(lambda x: (x * 2) - 1),
])

mnist_train = MNIST(root="./data", train=True, transform=train_tfm, download=True)
mnist_test  = MNIST(root="./data", train=False, transform=test_tfm, download=True)

100%|██████████| 9.91M/9.91M [00:02<00:00, 4.88MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 131kB/s]
100%|██████████| 1.65M/1.65M [00:01<00:00, 1.24MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 10.2MB/s]


In [ ]:
class SpectralNormParametrization(nn.Module):
    """
    Opacus-compatible Spectral Normalisation via torch.nn.utils.parametrize.

    Goal: bound the spectral norm (largest singular value σ) of each Conv2d / Linear
    weight to at most s_max.  When σ ≤ s_max the weight is unchanged; when σ > s_max
    the weight is divided by σ/s_max — a soft Lipschitz constraint.

    """
    def __init__(self, weight, s_max=1.0, n_power_iterations=1, eps=1e-12):
        super().__init__()
        self.s_max = float(s_max)
        self.n_power_iterations = n_power_iterations
        self.eps = eps

        out_dim = weight.shape[0]
        u = torch.randn(out_dim)
        u = u / (u.norm() + self.eps) # normalise to unit length

        self.register_parameter("sn_u", nn.Parameter(u, requires_grad=False))

    def forward(self, weight):
        out_dim = weight.shape[0]
        W_mat = weight.reshape(out_dim, -1)

        u = self.sn_u
        with torch.no_grad():
            for _ in range(self.n_power_iterations):
                v = torch.mv(W_mat.t(), u) # right singular vector
                v = v / (v.norm() + self.eps)
                u = torch.mv(W_mat, v) # left singular vector
                u = u / (u.norm() + self.eps)
            self.sn_u.copy_(u)

        v = torch.mv(W_mat.t(), u)
        v = v / (v.norm() + self.eps)
        sigma = torch.dot(u, torch.mv(W_mat, v)) # σ ≈ largest singular value

        #  Soft clamp: only divide when σ exceeds s_max 
        # scale ≥ 1 so weights are only shrunk, never amplified.
        scale = torch.clamp(sigma / self.s_max, min=1.0)
        return weight / scale

In [ ]:
# s_max policy: keep most layers at 1.0, loosen only some
def make_smax_fn(default=1.0, overrides=None):
    overrides = overrides or {}

    def smax_for(tag: str):
        if tag in overrides:
            return float(overrides[tag])
        for k, v in overrides.items():
            if tag.startswith(k):
                return float(v)
        return float(default)
    return smax_for

In [ ]:
def apply_spectral_norm(module, name="weight", s_max=1.0, n_power_iterations=1):
    """
    regs the SpectralNormParametrization to the specified module and weight name.
    """
    # get the current weight tensor to initialize the u vector correctly
    weight = getattr(module, name)

    # create the parametrization instance
    sn_param = SpectralNormParametrization(
        weight,
        s_max=s_max,
        n_power_iterations=n_power_iterations
    )

    # register it 
    parametrize.register_parametrization(module, name, sn_param)

In [ ]:
#  loosening policy
sn_smax_fn = make_smax_fn(
    default=1.0,
    overrides={
        "out_conv": 1.6,
        # skip conv helps preserve detail
        "res.skip": 1.2,
        # conditioning projection helps class/time conditioning under DP noise
        "time_proj": 1.2,
        "res.time": 1.2,
        "up.conv": 1.1,
    }
)

In [ ]:
def layer_wrapper(layer, use_sn: bool, tag: str, smax_fn, n_power_iterations=1):
    """
    if use_sn: apply SpectralNormParametrization to the layer.
    """
    # just return the layer if SN is disabled, used for sole DP-SGD training
    if not use_sn:
        return layer

    # get the s_max value for this layer tag
    s_max = float(smax_fn(tag))

    # apply SN using the parametrization API
    if isinstance(layer, (nn.Conv2d, nn.Linear)):
        apply_spectral_norm(
            layer,
            name="weight",
            s_max=s_max,
            n_power_iterations=n_power_iterations
        )
        return layer

    return layer

In [ ]:
class LipSiLU(nn.Module):
    # SiLU scaled by 1/max|SiLU'(x)| ≈ 1/1.0998 to bound the activation's Lipschitz constant at 1.
    def __init__(self, scale: float = 1/1.0998393):
        super().__init__()
        self.scale = float(scale)
    def forward(self, x):
        return self.scale * F.silu(x)

def get_time_embedding(time_steps, t_emb_dim, device):
    half_dim = t_emb_dim // 2
    emb = math.log(10000) / (half_dim - 1)
    emb = torch.exp(torch.arange(half_dim, device=device) * -emb)
    emb = time_steps[:, None].float() * emb[None, :]
    emb = torch.cat([torch.sin(emb), torch.cos(emb)], dim=1)
    if t_emb_dim % 2 == 1:
        emb = F.pad(emb, (0, 1, 0, 0))
    return emb

In [ ]:
class LipschitzGroupNorm(nn.Module):
    """
    Group Normalisation with a Lipschitz-safe denominator.

    Standard GroupNorm divides by the within-group standard deviation, which can
    approach zero and produce arbitrarily large outputs.  Clamping std ≥ 1.0 ensures the
    normalisation never amplifies activations, bounding the Lipschitz constant of this layer to ≤ 1.

    Note: affine scale/bias parameters (γ, β) are omitted here because adding
    learnable per-channel scales could break the unit-Lipschitz property.
    """
    def __init__(self, num_groups, num_channels):
        super().__init__()
        self.num_groups = num_groups
        self.num_channels = num_channels

    def forward(self, x):
        B, C, H, W = x.shape
        G = self.num_groups
        x = x.view(B, G, C // G, H, W)
        mean = x.mean(dim=[2, 3, 4], keepdim=True)
        var = x.var(dim=[2, 3, 4], keepdim=True, unbiased=False)
        std = (var + 1e-5).sqrt()
        denom = torch.clamp(std, min=1.0)    
        x_norm = (x - mean) / denom
        return x_norm.view(B, C, H, W)

class UpBlock(nn.Module):
    """Nearest-neighbour 2x upsample followed by a 3×3 conv."""
    def __init__(self, in_ch, out_ch, use_sn=False, smax_fn=None):
        super().__init__()
        self.up = nn.Upsample(scale_factor=2, mode="nearest")
        self.conv = layer_wrapper(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            use_sn=use_sn,
            tag="up.conv",
            smax_fn=smax_fn
        )

    def forward(self, x):
        return self.conv(self.up(x))

In [ ]:
class ResBlock(nn.Module):
    """
    Residual block with time-and-class conditioning and optional Lipschitz control.

    Processing pipeline per forward call:
      x  → conv1 → GroupNorm → time-conditioning (scale/shift) → act
         → conv2 → GroupNorm → act
         → (+ skip)

    Time conditioning:
      The fused (time + label) embedding is projected to (2 * out_channels) via
      time_mlp and split into `scale` and `shift`.  The scale is passed through
      tanh to bound it in [-1, 1], then applied as a multiplicative modulation
      (h * (1 + scale) + shift) — a form of FiLM conditioning.
    """
    def __init__(
        self,
        in_channels,
        out_channels,
        t_emb_dim,
        use_sn=False,
        use_lip_norm=False,
        alpha=0.4,
        n_power_iterations=1,
        eps=1e-8,
        smax_fn=None,
        tag_prefix="res"
    ):
        super().__init__()
        self.use_sn = use_sn
        self.use_lip_norm = use_lip_norm
        self.alpha = float(alpha)
        self.eps = float(eps)
        self.tag_prefix = tag_prefix

        self.act = LipSiLU() if use_lip_norm else nn.SiLU()

        self.conv1 = layer_wrapper(
            nn.Conv2d(in_channels, out_channels, 3, padding=1),
            use_sn=use_sn,
            tag=f"{tag_prefix}.conv1",
            smax_fn=smax_fn,
            n_power_iterations=n_power_iterations
        )
        self.conv2 = layer_wrapper(
            nn.Conv2d(out_channels, out_channels, 3, padding=1),
            use_sn=use_sn,
            tag=f"{tag_prefix}.conv2",
            smax_fn=smax_fn,
            n_power_iterations=n_power_iterations
        )

        self.time_mlp = nn.Sequential(
            self.act,
            layer_wrapper(
                nn.Linear(t_emb_dim, out_channels * 2),
                use_sn=use_sn,
                tag="res.time",
                smax_fn=smax_fn,
                n_power_iterations=n_power_iterations
            )
        )

        if in_channels != out_channels:
            self.skip_conv = layer_wrapper(
                nn.Conv2d(in_channels, out_channels, 1),
                use_sn=use_sn,
                tag="res.skip",
                smax_fn=smax_fn,
                n_power_iterations=n_power_iterations
            )
        else:
            self.skip_conv = nn.Identity()
            
        # Normalisation: Lipschitz-safe variant when SN is active
        if use_lip_norm:
            self.gn1 = LipschitzGroupNorm(num_groups=8, num_channels=out_channels)
            self.gn2 = LipschitzGroupNorm(num_groups=8, num_channels=out_channels)
        else:
            self.gn1 = nn.GroupNorm(8, out_channels, affine=True)
            self.gn2 = nn.GroupNorm(8, out_channels, affine=True)

    def forward(self, x, t_emb):
        h = self.conv1(x)
        h = self.gn1(h)

        #  FiLM-style time conditioning: project embedding → scale + shift
        time_emb = self.time_mlp(t_emb).unsqueeze(-1).unsqueeze(-1)
        scale, shift = time_emb.chunk(2, dim=1)
        scale = (torch.tanh(scale)+1)/2
        h = self.act(h * (scale) + shift)

        h = self.conv2(h)
        h = self.gn2(h)
        h = self.act(h)

        skip = self.skip_conv(x)

        if self.use_sn:
            # Weighted residual mix normalised by |a| + |1-a| to keep the
            # combined branch Lipschitz-bounded under spectral norm constraints.
            a = self.alpha
            mix = a * skip + (1.0 - a) * h
            denom = abs(a) + abs(1.0 - a) + self.eps
            return mix / denom

       # Standard 1/√2 normalisation: preserves variance for unit-variance branches
        return (h + skip) / math.sqrt(2)

In [ ]:
class CondDiffUNet(nn.Module):
    """
    Conditional Diffusion U-Net for MNIST (28×28 greyscale).
    CFG:
      Class index 10 is reserved as the 'null' (unconditional) label.
      At inference, two forward passes are done:
        v = v_uncond + guidance_scale * (v_cond - v_uncond)
    """
    def __init__(
        self,
        in_channels=1,
        base_channels=32,
        t_emb_dim=128,
        n_classes=10,
        use_sn=False,
        use_lip_norm=False,
        label_max_norm=10.0,
        smax_fn=None,
        sn_alpha=0.4,
    ):
        super().__init__()
        self.t_emb_dim = t_emb_dim
        self.n_classes = n_classes
        self.use_sn = use_sn

        # label embedding: not SN, but norm-limited
        # self.label_emb = nn.Embedding(n_classes, t_emb_dim, max_norm=float(label_max_norm))
        self.label_emb = nn.Embedding(n_classes, t_emb_dim)
        act = LipSiLU() if use_lip_norm else nn.SiLU()

        self.time_proj = nn.Sequential(
            layer_wrapper(nn.Linear(t_emb_dim, t_emb_dim), use_sn=use_sn, tag="time_proj", smax_fn=smax_fn),
            act,
            layer_wrapper(nn.Linear(t_emb_dim, t_emb_dim), use_sn=use_sn, tag="time_proj", smax_fn=smax_fn),
        )

        # Down
        self.down1 = ResBlock(in_channels, base_channels, t_emb_dim, use_sn, use_lip_norm,
                              alpha=sn_alpha, smax_fn=smax_fn, tag_prefix="down1")
        self.pool1 = layer_wrapper(
            nn.Conv2d(base_channels, base_channels, 4, stride=2, padding=1),
            use_sn=use_sn, tag="pool1", smax_fn=smax_fn
        )

        self.down2 = ResBlock(base_channels, base_channels*2, t_emb_dim, use_sn, use_lip_norm,
                              alpha=sn_alpha, smax_fn=smax_fn, tag_prefix="down2")
        self.pool2 = layer_wrapper(
            nn.Conv2d(base_channels*2, base_channels*2, 4, stride=2, padding=1),
            use_sn=use_sn, tag="pool2", smax_fn=smax_fn
        )

        self.down3 = ResBlock(base_channels*2, base_channels*4, t_emb_dim, use_sn, use_lip_norm,
                              alpha=sn_alpha, smax_fn=smax_fn, tag_prefix="down3")

        # Bottleneck
        self.bot1 = ResBlock(base_channels*4, base_channels*4, t_emb_dim, use_sn, use_lip_norm,
                             alpha=sn_alpha, smax_fn=smax_fn, tag_prefix="bot1")

        # Up
        self.up1 = UpBlock(base_channels*4, base_channels*2, use_sn=use_sn, smax_fn=smax_fn)
        self.up_res1 = ResBlock(base_channels*4, base_channels*2, t_emb_dim, use_sn, use_lip_norm,
                                alpha=sn_alpha, smax_fn=smax_fn, tag_prefix="up_res1")

        self.up2 = UpBlock(base_channels*2, base_channels, use_sn=use_sn, smax_fn=smax_fn)
        self.up_res2 = ResBlock(base_channels*2, base_channels, t_emb_dim, use_sn, use_lip_norm,
                                alpha=sn_alpha, smax_fn=smax_fn, tag_prefix="up_res2")

        #final 3×3 conv — predicts v (v-parameterisation target)
        self.out_conv = layer_wrapper(
            nn.Conv2d(base_channels, in_channels, 3, padding=1),
            use_sn=use_sn, tag="out_conv", smax_fn=smax_fn
        )

    def forward(self, x, t, y):
        t_emb = get_time_embedding(t, self.t_emb_dim, x.device)
        y_emb = self.label_emb(y)
        emb = self.time_proj(t_emb + y_emb)

        #encoder 
        x1 = self.down1(x, emb)
        x2 = self.pool1(x1)
        x2 = self.down2(x2, emb)
        x3 = self.pool2(x2)
        x3 = self.down3(x3, emb)
        bn = self.bot1(x3, emb) #bottleneck

        # decoder
        u1 = self.up1(bn)
        u1 = torch.cat([u1, x2], dim=1) / math.sqrt(2)
        u1 = self.up_res1(u1, emb)
        u2 = self.up2(u1)
        u2 = torch.cat([u2, x1], dim=1) / math.sqrt(2)
        u2 = self.up_res2(u2, emb)

        return self.out_conv(u2)

In [ ]:
class DDPMConfig:
     # hyperparams for the DDPM noise schedule
    def __init__(self, timesteps=1000, beta_start=1e-4, beta_end=0.02):
        self.T = timesteps
        self.beta_start = beta_start
        self.beta_end = beta_end

def make_ddpm_schedule(cfg: DDPMConfig, device):
    """
    precompute the linear beta schedule and derived quantities.
    """
    betas = torch.linspace(cfg.beta_start, cfg.beta_end, cfg.T, device=device)
    alphas = 1.0 - betas
    alpha_bars = torch.cumprod(alphas, dim=0)
    return betas, alphas, alpha_bars

@torch.no_grad()
def v_to_eps_and_x0(x_t, t, v, alpha_bars):
    """
    Recover the noise prediction ε and clean image x₀ from a v-prediction.

    v-param.:
        v = √ᾱₜ · ε − √(1−ᾱₜ) · x₀

    Inverting these relations gives:
        x₀ = √ᾱₜ · xₜ − √(1−ᾱₜ) · v
        ε  = √(1−ᾱₜ) · xₜ + √ᾱₜ  · v

    Predicting v instead of ε gives better-conditioned gradients at both
    low and high noise levels.
    """
    ab = alpha_bars[t].view(-1, 1, 1, 1)
    sqrt_ab = torch.sqrt(ab)
    sqrt_1mab = torch.sqrt(1.0 - ab)

    x0  = sqrt_ab * x_t - sqrt_1mab * v
    eps = sqrt_1mab * x_t + sqrt_ab * v
    return eps, x0

def compute_v_target(x0, eps, t, alpha_bars):
    """
    Compute the v-prediction target given clean image x₀ and sampled noise ε.

        v_target = √ᾱₜ · ε − √(1−ᾱₜ) · x₀
    """
    ab = alpha_bars[t].view(-1, 1, 1, 1)
    return torch.sqrt(ab) * eps - torch.sqrt(1.0 - ab) * x0

In [ ]:
class EMA:
    """
    Exponential Moving Average of model parameters.

    Maintains a shadow copy of each trainable parameter updated each step as:
        shadow = decay * shadow + (1 - decay) * param

    The EMA weights are used exclusively for sampling, they are not used during
    the training backward pass, so they do not affect DP accounting.
    """
    def __init__(self, model: nn.Module, decay=0.999):
        self.decay = decay
        self.shadow = {}
        self._init_shadow(model)

    def _init_shadow(self, model):
        for name, p in model.named_parameters():
            if p.requires_grad:
                self.shadow[name] = p.detach().clone()

    @torch.no_grad()
    def update(self, model):
        for name, p in model.named_parameters():
            if not p.requires_grad:
                continue
            self.shadow[name].mul_(self.decay).add_(p.detach(), alpha=(1.0 - self.decay))

    @torch.no_grad()
    def copy_to(self, model):
        for name, p in model.named_parameters():
            if name in self.shadow:
                p.data.copy_(self.shadow[name].data)

    def state_dict(self):
        return {"decay": self.decay, "shadow": self.shadow}

    def load_state_dict(self, state):
        self.decay = state["decay"]
        self.shadow = state["shadow"]

In [ ]:
# DP diagnostics: clipping ratio & grad norm distribution
def _collect_per_sample_grad_norms(model):
    norms_sq = None
    for p in model.parameters():
        gs = getattr(p, "grad_sample", None)
        if gs is None:
            continue
        gs_flat = gs.reshape(gs.shape[0], -1)
        cur = (gs_flat ** 2).sum(dim=1)  # [B]
        norms_sq = cur if norms_sq is None else (norms_sq + cur)
    if norms_sq is None:
        raise RuntimeError("No grad_sample found. Ensure Opacus make_private() was used and backward() called.")
    return torch.sqrt(norms_sq + 1e-12)

def grad_norm_stats(norms, clip_C):
    n = norms.detach()
    return {
        "clip_frac": (n > clip_C).float().mean().item(),
        "mean": n.mean().item(),
        "median": n.median().item(),
        "p90": torch.quantile(n, 0.90).item(),
        "p99": torch.quantile(n, 0.99).item(),
        "max": n.max().item(),
    }

In [ ]:
def save_norm_hist(norms: torch.Tensor, clip_C: float, out_path: str, bins: int = 60):
    n = norms.detach().float().cpu().numpy()
    plt.figure()
    plt.hist(n, bins=bins)
    plt.axvline(clip_C)   # threshold line at C
    plt.title("Per-sample grad norm histogram")
    plt.xlabel("||g_i||2")
    plt.ylabel("count")
    plt.tight_layout()
    plt.savefig(out_path, dpi=160)
    plt.close()

def plot_dp_diagnostics(df_diag: pd.DataFrame, clip_C: float, out_dir: str):
    """
    Generate two diagnostic plots and save them to out_dir:
      1. Clipping fraction over training steps — tracks how often DP clips
      2. Percentile curves of per-sample gradient norms — shows norm distribution
    """
    os.makedirs(out_dir, exist_ok=True)

    # clipping fraction plot
    plt.figure()
    plt.plot(df_diag["step"], df_diag["clip_frac"])
    plt.title("Clipping fraction over steps")
    plt.xlabel("step")
    plt.ylabel("fraction(||g_i|| > C)")
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, "clip_fraction_vs_step.png"), dpi=160)
    plt.close()

    # Plot 2: norm percentile curves
    plt.figure()
    plt.plot(df_diag["step"], df_diag["mean"], label="mean")
    plt.plot(df_diag["step"], df_diag["median"], label="median")
    plt.plot(df_diag["step"], df_diag["p90"], label="p90")
    plt.plot(df_diag["step"], df_diag["p99"], label="p99")
    plt.plot(df_diag["step"], df_diag["max"], label="max")
    plt.axhline(clip_C)
    plt.title("Per-sample grad norm stats over steps")
    plt.xlabel("step")
    plt.ylabel("||g_i||2")
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, "grad_norm_stats_vs_step.png"), dpi=160)
    plt.close()

### DP training runner

In [ ]:
class DPConditionalDiffusionRunner:
    """
    End-to-end training runner for a DP-SGD conditional diffusion model.
    
    Poisson sampling: each example is included in a batch independently
    with probability batch_size/N.  This is required for the standard
    Rényi-DP / moments-accountant privacy analysis.
  
    K-multiplicity (noise multiplicity):
      For each logical batch, K=16 independent forward-backward passes are
      performed with different timestep and noise samples.  The per-sample
      grad_sample tensors are accumulated and averaged before the DP clipping
      step.  This reduces the Monte Carlo variance in the stochastic loss
      estimate without changing the privacy cost (same batch, same step).
    """
    def __init__(
        self,
        exp_name: str,
        model: nn.Module,
        cfg: DDPMConfig,
        train_dataset,
        test_dataset,
        batch_size=4096,
        lr=3e-4,
        epochs=300,
        noise_multiplicity_K=16,
        ema_decay=0.999,
        num_workers=2,

        dp_noise_multiplier=1.2,
        dp_max_grad_norm=1.0,
        dp_delta=1e-5,
        poisson_sampling=True,
        secure_mode=False,

        diag_every_steps=10,
        save_hist_every_steps=200,
    ):
        self.exp_name = exp_name
        self.exp_dir = make_exp_dir(exp_name)

        self.cfg = cfg
        self.betas, self.alphas, self.alpha_bars = make_ddpm_schedule(cfg, device=device)

        self.batch_size = batch_size
        self.lr = lr
        self.epochs = epochs
        self.K = int(noise_multiplicity_K)

        self.dp_noise_multiplier = float(dp_noise_multiplier)
        self.dp_max_grad_norm = float(dp_max_grad_norm)
        self.dp_delta = float(dp_delta)
        self.poisson_sampling = bool(poisson_sampling)

        self.diag_every_steps = int(diag_every_steps)
        self.save_hist_every_steps = int(save_hist_every_steps)

        # data loaders
        self.train_loader = DataLoader(
            train_dataset, batch_size=batch_size, shuffle=True, drop_last=True,
            num_workers=num_workers, pin_memory=True
        )
        self.test_loader = DataLoader(
            test_dataset, batch_size=256, shuffle=False, drop_last=False,
            num_workers=num_workers, pin_memory=True
        )

        # model + optimizer + EMA
        self.model = model.to(device)
        self.ema = EMA(self.model, decay=ema_decay)
        self.optimizer = torch.optim.Adam(self.model.parameters(), lr=self.lr)
        self.criterion = nn.MSELoss(reduction="none")

        if not ModuleValidator.is_valid(self.model):
            self.model = ModuleValidator.fix(self.model)

        self.privacy_engine = PrivacyEngine(secure_mode=secure_mode)

        # force_functorch=True is needed here because some custom layers
        # (e.g. SpectralNormParametrization) aren't supported by Opacus's
        # default per-sample gradient implementation.
        self.model = GradSampleModule(
            self.model,
            strict=False,
            force_functorch=True,
            batch_first=True,
            loss_reduction="mean"
        )

        self.model, self.optimizer, self.train_loader = self.privacy_engine.make_private(
            module=self.model,
            optimizer=self.optimizer,
            data_loader=self.train_loader,
            noise_multiplier=self.dp_noise_multiplier,
            max_grad_norm=self.dp_max_grad_norm,
            poisson_sampling=self.poisson_sampling,
        )

        # save config
        save_json({
            "exp_name": exp_name,
            "timesteps": cfg.T,
            "beta_start": cfg.beta_start,
            "beta_end": cfg.beta_end,
            "batch_size": batch_size,
            "lr": lr,
            "epochs": epochs,
            "noise_multiplicity_K": self.K,
            "ema_decay": ema_decay,
            "device": str(device),
            "dp_noise_multiplier": self.dp_noise_multiplier,
            "dp_max_grad_norm": self.dp_max_grad_norm,
            "dp_delta": self.dp_delta,
            "poisson_sampling": self.poisson_sampling,
        }, os.path.join(self.exp_dir, "config.json"))

    def _epsilon(self):
        """Accumulated epsilon so far"""
        return self.privacy_engine.accountant.get_epsilon(self.dp_delta)

    def train(self, log_every=50):
        metrics = []
        for ep in range(1, self.epochs + 1):
            self.model.train()
            for step, (x, y) in enumerate(self.train_loader, 1):
                x = x.to(device)
                y = y.to(device)

                t = torch.randint(0, self.cfg.T, (x.shape[0],), device=device)

                noise = torch.randn_like(x)
                x_t = self._diffuse(x, t, noise)

                # v-prediction target is shared across all K noise draws
                v_target = compute_v_target(x, noise, t, self.alpha_bars)

                # Noise multiplicity: draw K independent noise samples for the same batch
                # and average the losses. This reduces gradient variance without extra
                # privacy cost — each step still counts as a single DP mechanism call.
                loss_total = 0.0
                for k in range(self.K):
                    noise_k = torch.randn_like(x)
                    x_t_k = self._diffuse(x, t, noise_k)
                    pred_v_k = self.model(x_t_k, t, y)
                    loss_k = self.criterion(pred_v_k, v_target).mean()
                    loss_total = loss_total + loss_k

                loss = loss_total / self.K

                self.optimizer.zero_grad(set_to_none=True)
                loss.backward()

                # DP diagnostics
                if step % self.diag_every_steps == 0:
                    norms = _collect_per_sample_grad_norms(self.model)
                    stats = grad_norm_stats(norms, self.dp_max_grad_norm)
                    metrics.append({
                        "epoch": ep,
                        "step": step + (ep - 1) * len(self.train_loader),
                        **stats
                    })

                    if step % self.save_hist_every_steps == 0:
                        save_norm_hist(
                            norms,
                            self.dp_max_grad_norm,
                            os.path.join(self.exp_dir, "diagnostics", f"hist_ep{ep}_step{step}.png")
                        )

                self.optimizer.step()
                self.ema.update(self.model)

                if step % log_every == 0:
                    eps = self._epsilon()
                    print(f"Epoch {ep}/{self.epochs} | Step {step}/{len(self.train_loader)} | Loss {loss:.4f} | Eps {eps:.2f}")

            print(f"Epoch {ep}/{self.epochs} complete. Current epsilon: {self._epsilon():.2f}")

        df_metrics = pd.DataFrame(metrics)
        plot_dp_diagnostics(df_metrics, self.dp_max_grad_norm, os.path.join(self.exp_dir, "diagnostics"))
        df_metrics.to_csv(os.path.join(self.exp_dir, "diagnostics", "metrics.csv"), index=False)

        return df_metrics

    def _diffuse(self, x0, t, noise):
        ab = self.alpha_bars[t].view(-1, 1, 1, 1)
        return torch.sqrt(ab) * x0 + torch.sqrt(1.0 - ab) * noise

    @torch.no_grad()
    def sample(self, n_samples, batch_size=256, guidance_scale=1.0):
        self.ema.copy_to(self.model)
        self.model.eval()

        samples = []
        for y in tqdm(range(0, 10)):
            y_batch = torch.full((batch_size,), y, dtype=torch.long, device=device)
            x_t = torch.randn(batch_size, 1, 28, 28, device=device)

            for t in range(self.cfg.T - 1, -1, -1):
                t_batch = torch.full((batch_size,), t, dtype=torch.long, device=device)

                pred_v = self.model(x_t, t_batch, y_batch)

                # Convert v-prediction to epsilon for the DDPM reverse step
                ab = self.alpha_bars[t]
                sqrt_ab = torch.sqrt(ab)
                sqrt_1mab = torch.sqrt(1.0 - ab)
                eps_pred = sqrt_1mab * x_t + sqrt_ab * pred_v

                alpha_t = self.alphas[t]
                beta_t = self.betas[t]

                if t > 0:
                    noise = torch.randn_like(x_t)
                    sigma_t = torch.sqrt((1.0 - alpha_t) / (1.0 - ab) * beta_t)
                    x_t = (x_t - beta_t / sqrt_1mab * eps_pred) / torch.sqrt(alpha_t) + sigma_t * noise
                else:
                    x_t = (x_t - beta_t / sqrt_1mab * eps_pred) / torch.sqrt(alpha_t)

            x_t = torch.clamp(x_t, -1, 1)
            samples.append(x_t.cpu())

        return torch.cat(samples, dim=0)

    def generate_synthetic_dataset(self, n_samples=60000, batch_size=256, save_pngs=True, guidance_scale=1.0):
        n_batches = (n_samples + batch_size - 1) // batch_size
        all_images_u8 = []
        all_labels = []

        for y in tqdm(range(0, 10)):
            samples_for_class = (n_samples // 10)
            remaining = n_samples % 10
            if y < remaining:
                samples_for_class += 1

            for _ in tqdm(range((samples_for_class + batch_size - 1) // batch_size), leave=False):
                batch_size_actual = min(batch_size, samples_for_class - len([x for x in all_labels if x == y]))

                y_batch = torch.full((batch_size_actual,), y, dtype=torch.long, device=device)
                x_t = torch.randn(batch_size_actual, 1, 28, 28, device=device)

                for t in range(self.cfg.T - 1, -1, -1):
                    t_batch = torch.full((batch_size_actual,), t, dtype=torch.long, device=device)
                    pred_v = self.model(x_t, t_batch, y_batch)

                    ab = self.alpha_bars[t]
                    sqrt_ab = torch.sqrt(ab)
                    sqrt_1mab = torch.sqrt(1.0 - ab)
                    eps_pred = sqrt_1mab * x_t + sqrt_ab * pred_v

                    alpha_t = self.alphas[t]
                    beta_t = self.betas[t]

                    if t > 0:
                        noise = torch.randn_like(x_t)
                        sigma_t = torch.sqrt((1.0 - alpha_t) / (1.0 - ab) * beta_t)
                        x_t = (x_t - beta_t / sqrt_1mab * eps_pred) / torch.sqrt(alpha_t) + sigma_t * noise
                    else:
                        x_t = (x_t - beta_t / sqrt_1mab * eps_pred) / torch.sqrt(alpha_t)

                x_t = torch.clamp(x_t, -1, 1)
                images_u8 = ((x_t + 1) / 2 * 255).byte().cpu()
                all_images_u8.append(images_u8)
                all_labels.extend([y] * batch_size_actual)

        all_images_u8 = torch.cat(all_images_u8, dim=0)[:n_samples]
        all_labels = torch.tensor(all_labels[:n_samples])

        synth_pt = os.path.join(self.exp_dir, "synthetic_data", "synth.pt")
        torch.save({
            "images_u8": all_images_u8,
            "labels": all_labels
        }, synth_pt)

        if save_pngs:
            pngs_dir = os.path.join(self.exp_dir, "synthetic_data", "pngs")
            os.makedirs(pngs_dir, exist_ok=True)
            for i, (img, label) in enumerate(zip(all_images_u8, all_labels)):
                save_image(img.float() / 255.0, os.path.join(pngs_dir, f"synth_{i:06d}_y{label}.png"))

        return synth_pt

    def export_png_folder_for_fid_is(self, n_samples=20000, batch_size=256, folder_name="gen", guidance_scale=1.0):
        out_folder = os.path.join(self.exp_dir, "fid_is", folder_name)
        os.makedirs(out_folder, exist_ok=True)

        idx = 0
        for y in tqdm(range(0, 10)):
            samples_per_class = n_samples // 10

            for _ in tqdm(range((samples_per_class + batch_size - 1) // batch_size), leave=False):
                batch_size_actual = min(batch_size, samples_per_class)

                y_batch = torch.full((batch_size_actual,), y, dtype=torch.long, device=device)
                x_t = torch.randn(batch_size_actual, 1, 28, 28, device=device)

                for t in range(self.cfg.T - 1, -1, -1):
                    t_batch = torch.full((batch_size_actual,), t, dtype=torch.long, device=device)
                    pred_v = self.model(x_t, t_batch, y_batch)

                    ab = self.alpha_bars[t]
                    sqrt_ab = torch.sqrt(ab)
                    sqrt_1mab = torch.sqrt(1.0 - ab)
                    eps_pred = sqrt_1mab * x_t + sqrt_ab * pred_v

                    alpha_t = self.alphas[t]
                    beta_t = self.betas[t]

                    if t > 0:
                        noise = torch.randn_like(x_t)
                        sigma_t = torch.sqrt((1.0 - alpha_t) / (1.0 - ab) * beta_t)
                        x_t = (x_t - beta_t / sqrt_1mab * eps_pred) / torch.sqrt(alpha_t) + sigma_t * noise
                    else:
                        x_t = (x_t - beta_t / sqrt_1mab * eps_pred) / torch.sqrt(alpha_t)

                x_t = torch.clamp(x_t, -1, 1)
                images_u8 = ((x_t + 1) / 2 * 255).byte().cpu()

                for img in images_u8:
                    save_image(img.float() / 255.0, os.path.join(out_folder, f"gen_{idx:06d}.png"))
                    idx += 1

        return out_folder

In [ ]:
def prepare_real_images_folder(dataset, output_folder, num_images=20000):
    os.makedirs(output_folder, exist_ok=True)
    existing = len([f for f in os.listdir(output_folder) if f.endswith(".png")])
    if existing >= num_images:
        print(f"Real images folder already has {existing} PNGs (>= {num_images}). Skipping.")
        return output_folder

    loader = DataLoader(dataset, batch_size=1, shuffle=True, drop_last=False)
    i = 0
    for img, _ in tqdm(loader, total=num_images, desc="Saving real images"):
        if i >= num_images:
            break
        img = img.squeeze(0)
        img01 = (img + 1) / 2
        save_image(img01, os.path.join(output_folder, f"real_{i:06d}.png"))
        i += 1

    print(f"Saved {i} real images to {output_folder}")
    return output_folder

In [ ]:
def compute_fid_and_is(gen_dir, real_dir, use_cuda=True):
    metrics = calculate_metrics(
        input1=gen_dir,
        input2=real_dir,
        cuda=use_cuda,
        fid=True,
        isc=True,
        verbose=False
    )
    return {
        "FID": float(metrics["frechet_inception_distance"]),
        "IS_mean": float(metrics["inception_score_mean"]),
        "IS_std": float(metrics["inception_score_std"]),
    }

In [ ]:
class SynthMNISTDataset(torch.utils.data.Dataset):
    def __init__(self, pt_path):
        d = torch.load(pt_path, map_location="cpu")
        self.images_u8 = d["images_u8"]
        self.labels = d["labels"]
    def __len__(self):
        return self.labels.shape[0]
    def __getitem__(self, idx):
        x = self.images_u8[idx].float() / 255.0
        x = (x * 2) - 1
        y = self.labels[idx].long()
        return x, y

class MLPClassifier(nn.Module):
    def __init__(self, hidden=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(28*28, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Linear(hidden, 10)
        )
    def forward(self, x):
        return self.net(x.view(x.size(0), -1))


In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.fc = nn.Sequential(
            nn.Linear(64*7*7, 128), nn.ReLU(),
            nn.Linear(128, 10)
        )
    def forward(self, x):
        h = self.conv(x)
        h = h.view(h.size(0), -1)
        return self.fc(h)

In [ ]:
@torch.no_grad()
def eval_accuracy(model, loader, device):
    model.eval()
    correct, total = 0, 0
    for x, y in loader:
        x = x.to(device); y = y.to(device)
        pred = model(x).argmax(dim=1)
        correct += (pred == y).sum().item()
        total += y.numel()
    return correct / total

def train_classifier(model, train_loader, test_loader, device, epochs=10, lr=1e-3):
    model = model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    ce = nn.CrossEntropyLoss()
    for ep in range(1, epochs+1):
        model.train()
        for x, y in tqdm(train_loader, desc=f"[clf] ep {ep}/{epochs}", leave=False):
            x = x.to(device); y = y.to(device)
            loss = ce(model(x), y)
            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()
        acc = eval_accuracy(model, test_loader, device)
        print(f"  ep {ep}: test_acc={acc:.4f}")
    return eval_accuracy(model, test_loader, device)

In [ ]:
def downstream_eval_nn(synth_pt_path, mnist_test_ds, out_dir, device, batch_size=256):
    os.makedirs(out_dir, exist_ok=True)
    synth_ds = SynthMNISTDataset(synth_pt_path)
    synth_loader = DataLoader(synth_ds, batch_size=batch_size, shuffle=True, drop_last=False, num_workers=2)
    test_loader = DataLoader(mnist_test_ds, batch_size=batch_size, shuffle=False, drop_last=False, num_workers=2)

    results = {}
    print("\n[downstream NN] MLP")
    results["mlp"] = float(train_classifier(MLPClassifier(), synth_loader, test_loader, device=device, epochs=10, lr=1e-3))
    print("\n[downstream NN] CNN")
    results["cnn"] = float(train_classifier(SimpleCNN(), synth_loader, test_loader, device=device, epochs=10, lr=1e-3))

    save_json(results, os.path.join(out_dir, "downstream_accuracy_nn.json"))
    pd.DataFrame([results]).to_csv(os.path.join(out_dir, "downstream_accuracy_nn.csv"), index=False)
    return results

In [ ]:
def load_synth_pt_to_numpy(synth_pt_path):
    d = torch.load(synth_pt_path, map_location="cpu")
    X = d["images_u8"].numpy().astype(np.float32) / 255.0
    X = X.reshape(X.shape[0], -1)
    y = d["labels"].numpy().astype(np.int64)
    return X, y

def load_real_mnist_test_to_numpy(mnist_test_ds):
    loader = DataLoader(mnist_test_ds, batch_size=512, shuffle=False)
    X_list, y_list = [], []
    for x, y in loader:
        x = ((x + 1) / 2).numpy().astype(np.float32)
        X_list.append(x.reshape(x.shape[0], -1))
        y_list.append(y.numpy().astype(np.int64))
    return np.concatenate(X_list, axis=0), np.concatenate(y_list, axis=0)

In [ ]:
def downstream_eval_sklearn(synth_pt_path, mnist_test_ds, out_dir, random_state=42):
    os.makedirs(out_dir, exist_ok=True)
    X_train, y_train = load_synth_pt_to_numpy(synth_pt_path)
    X_test, y_test = load_real_mnist_test_to_numpy(mnist_test_ds)

    results = {}

    print("\n[sklearn] LogisticRegression...")
    logreg = LogisticRegression(max_iter=300, solver="lbfgs", n_jobs=-1)
    logreg.fit(X_train, y_train)
    results["logreg"] = float(accuracy_score(y_test, logreg.predict(X_test)))
    print("  test_acc:", results["logreg"])

    print("\n[sklearn] DecisionTree...")
    dt = DecisionTreeClassifier(random_state=random_state)
    dt.fit(X_train, y_train)
    results["decision_tree"] = float(accuracy_score(y_test, dt.predict(X_test)))
    print("  test_acc:", results["decision_tree"])

    print("\n[sklearn] RandomForest...")
    rf = RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=random_state)
    rf.fit(X_train, y_train)
    results["random_forest"] = float(accuracy_score(y_test, rf.predict(X_test)))
    print("  test_acc:", results["random_forest"])

    save_json(results, os.path.join(out_dir, "downstream_accuracy_sklearn.json"))
    pd.DataFrame([results]).to_csv(os.path.join(out_dir, "downstream_accuracy_sklearn.csv"), index=False)
    return results

In [ ]:
def evaluate_runner(runner, exp_name, mnist_train, mnist_test, RUN_DIR, device,
                    synth_n=60000, fid_is_n=20000):
    # synth
    synth_pt = runner.generate_synthetic_dataset(n_samples=synth_n, batch_size=256, save_pngs=False, guidance_scale=1.0)

    # real folder
    REAL_DIR = os.path.join(RUN_DIR, "fid_is_real_mnist_train_20k")
    prepare_real_images_folder(mnist_train, REAL_DIR, num_images=fid_is_n)

    # gen folder
    gen_dir = runner.export_png_folder_for_fid_is(n_samples=fid_is_n, batch_size=256, folder_name="gen_20k", guidance_scale=1.0)

    # fidi/is
    fidis = compute_fid_and_is(gen_dir, REAL_DIR, use_cuda=torch.cuda.is_available())
    save_json(fidis, os.path.join(runner.exp_dir, "metrics", "fid_is_20k.json"))

    metrics_dir = os.path.join(runner.exp_dir, "metrics")
    nn_res  = downstream_eval_nn(synth_pt, mnist_test, metrics_dir, device=device)
    skl_res = downstream_eval_sklearn(synth_pt, mnist_test, metrics_dir)

    combined = {
        "exp_name": exp_name,
        "epsilon_final": runner._epsilon(),
        "FID": fidis["FID"],
        "IS_mean": fidis["IS_mean"],
        "IS_std": fidis["IS_std"],
        "downstream": {
            "mlp": nn_res["mlp"],
            "cnn": nn_res["cnn"],
            "logreg": skl_res["logreg"],
            "decision_tree": skl_res["decision_tree"],
            "random_forest": skl_res["random_forest"],
        }
    }
    save_json(combined, os.path.join(metrics_dir, "all_metrics.json"))
    return combined

In [ ]:
ddpm_cfg = DDPMConfig(timesteps=1000, beta_start=1e-4, beta_end=0.02)

# Core training hyperparameters. The noise multiplier is set separately below
# based on the target epsilon budget using the Opacus accountant.
BATCH_SIZE = 2048
EPOCHS = 100
LR = 3e-4
K_MULT = 16       # number of noise draws per batch (noise multiplicity)
EMA_DECAY = 0.999

DP_MAX_GRAD_NORM = 1.0
DP_DELTA = 1e-5

In [ ]:
from opacus.accountants.utils import get_noise_multiplier

# Back-calculate the noise multiplier needed to hit a target epsilon at the given
# sample rate and number of training steps. useful for choosing sigma before training.
DATASET_SIZE = 60000
DELTA = DP_DELTA

sample_rate = BATCH_SIZE / DATASET_SIZE

print(f"Stats: Epochs={EPOCHS}, Batch={BATCH_SIZE}, Rate={sample_rate:.4f}")
print("-" * 40)

for target_eps in [1, 10, 50]:
    noise_mult = get_noise_multiplier(
        target_epsilon=target_eps,
        target_delta=DELTA,
        sample_rate=sample_rate,
        epochs=EPOCHS,
    )
    print(f" Epsilon = {target_eps:<2} ->  DP_NOISE_MULT = {noise_mult:.4f}")
    
# get_noise_multiplier() uses the Rényi-DP accountant to find the noise
# multiplier σ that achieves (ε=10, δ=1e-5) after 100 epochs of Poisson
# sampling with batch_size=2048 over N=60000 samples.
# The resulting σ=1.2073 balances privacy (ε≈10) against utility.

In [ ]:
# Targets epsilon ≈ 10 at delta=1e-5 with batch=2048 on 60k samples over 100 epochs.
DP_NOISE_MULT = 1.2073
DP_MAX_GRAD_NORM = 1.0

### Experiment 1

In [ ]:
# Baseline run: plain DP-DDPM without spectral norm or Lipschitz normalisation.
# Serves as the reference point for comparing Lip-constrained variants.
exp_dp_base = f"DP_BASELINE_MSE_K{K_MULT}_B{BATCH_SIZE}_nm{DP_NOISE_MULT}_cn{DP_MAX_GRAD_NORM}"
model_dp_base = CondDiffUNet(
    n_classes=11,
    use_sn=False,
    use_lip_norm=False,
    label_max_norm=10.0,
    smax_fn=None,
    sn_alpha=0.4
).to(device)

runner_dp_base = DPConditionalDiffusionRunner(
    exp_name=exp_dp_base,
    model=model_dp_base,
    cfg=ddpm_cfg,
    train_dataset=mnist_train,
    test_dataset=mnist_test,
    batch_size=BATCH_SIZE,
    lr=LR,
    epochs=EPOCHS,
    noise_multiplicity_K=K_MULT,
    ema_decay=EMA_DECAY,
    dp_noise_multiplier=DP_NOISE_MULT,
    dp_max_grad_norm=DP_MAX_GRAD_NORM,
    dp_delta=DP_DELTA,
    poisson_sampling=True,
    diag_every_steps=10,
    save_hist_every_steps=200,
)

hist_base = runner_dp_base.train(log_every=50)
print(hist_base.tail())

In [ ]:
metrics_base = evaluate_runner(runner_dp_base, exp_dp_base, mnist_train, mnist_test, RUN_DIR, device)

Generating with Guidance Scale w=1.0...


[DP_BASELINE_MSE_K16_B2048_nm1.2073_cn3.0] generating 60000: 100%|██████████| 235/235 [48:52<00:00, 12.48s/it]


[DP_BASELINE_MSE_K16_B2048_nm1.2073_cn3.0] saved synthetic dataset to: /content/drive/MyDrive/diffusion_experiments/14.01.2026/DP_BASELINE_MSE_K16_B2048_nm1.2073_cn3.0/synthetic_data/synth_mnist_60000_w1.0.pt


Saving real images: 100%|██████████| 20000/20000 [01:44<00:00, 192.06it/s]


Saved 20000 real images to /content/drive/MyDrive/diffusion_experiments/14.01.2026/fid_is_real_mnist_train_20k
Exporting PNGs with Guidance Scale w=1.0...


...

  ep 6: test_acc=0.9486


  ep 7: test_acc=0.9527


  ep 8: test_acc=0.9493


  ep 9: test_acc=0.9489


  ep 10: test_acc=0.9499

[sklearn] LogisticRegression...
  test_acc: 0.8828

[sklearn] DecisionTree...
  test_acc: 0.7897

[sklearn] RandomForest...
  test_acc: 0.9176
